# Experiment: trial_branch retrieval + existing semantic classifier

Run from `project/` on TIP **after 07_classifier** with the same ENV and labels.
This notebook queries PATSTAT: it reproduces the four trial_branch blocks exactly,
using the frozen source at `c509b9b5e7de13762c832c47ea3d0ee5019b2802`.
Source: https://github.com/polito-IM26/im26-pw-team-05/tree/c509b9b5e7de13762c832c47ea3d0ee5019b2802

The classifier is the existing model trained on the broad labelled pool, using
Patent SBERT embeddings + semantic/BM25 features. We do **not** retrain on the
narrow subset: this experiment isolates the effect of narrow retrieval.

All four policies are evaluated on the SAME labelled families with OOF scores.
Narrow-query misses count as rejections, including missed positives. These are
exploratory sample metrics, not independent test or population estimates.
The query's own date/member conditions are preserved, not approximated from titles.

Missing scores outside the broad pool are exported explicitly; their hybrid
classification is pending. Production volumes with missing scores are incomplete.
No original labels, model, or sector decisions are modified.
Rollback: switch to `experiment/rule-a-measurement` (f019a4d).


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from epo.tipdata.patstat import PatstatClient
from src.trial_snapshot.sector_config import load_sector_config
from src.trial_snapshot.query_builders import SectorQueryBuilder
from src.policy_metrics import binary_metrics
from src.labels import load_labels

ENV = "PROD"
OUTPUT_DIR = Path("outputs/step0")
EXPERIMENT_DIR = OUTPUT_DIR / f"narrow_hybrid_{ENV}"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
THRESHOLDS = [0.30, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

def read_table(stem):
    p = OUTPUT_DIR / f"{stem}_{ENV}.parquet"
    return pd.read_parquet(p) if p.exists() else pd.read_csv(p.with_suffix(".csv"))

cv = pd.read_csv(OUTPUT_DIR / f"classifier_cv_{ENV}.csv")
scored = read_table("classified")
labels = load_labels(Path("data/labels"))
if cv["docdb_family_id"].duplicated().any() or scored["docdb_family_id"].duplicated().any():
    raise ValueError("Duplicate family IDs in 07 outputs")
if cv["p_cv"].isna().any():
    raise ValueError("Missing OOF scores; rerun 07")
# Reject stale outputs if labels changed after running 07.
check = cv.merge(labels[["docdb_family_id", "relevance"]], on="docdb_family_id", how="left", validate="one_to_one")
truth = pd.to_numeric(check["category"], errors="raise").eq(1)
if check["relevance"].isna().any() or not truth.eq(check["relevance"].eq(1)).all():
    raise ValueError("Labels changed or conflict: rerun 07 before this experiment")


## 1. Exact narrow-query membership
The full query runs on TIP; only family IDs and four membership flags are downloaded.
The snapshot is pinned for reproducibility and is not automatically updated.


In [ ]:
cfg = load_sector_config(Path("src/trial_snapshot/sector.json"))
patstat = PatstatClient(env=ENV)
db = patstat.orm()
member_query = SectorQueryBuilder(db, cfg).family_membership()
membership = patstat.df(db.query(member_query))
if membership["docdb_family_id"].duplicated().any():
    raise ValueError("Narrow membership must have one row per family")
membership.to_csv(EXPERIMENT_DIR / "membership.csv", index=False)
narrow_ids = set(membership["docdb_family_id"])
print(f"Exact trial_branch query: {len(narrow_ids):,} families")


## 2. Same-denominator OOF comparison
`narrow_query_only` accepts narrow members regardless of score.
`narrow_plus_classifier` requires narrow membership AND score >= threshold.
Broad policies reproduce 07. Thresholds are diagnostics, not tuned test results.


In [ ]:
evaluated = check.merge(scored[["docdb_family_id", "tier"]], on="docdb_family_id", how="left", validate="one_to_one")
if evaluated["tier"].isna().any():
    raise ValueError("CV and classified outputs do not match; rerun 07")
y = evaluated["relevance"].eq(1)
narrow = evaluated["docdb_family_id"].isin(narrow_ids)
core = evaluated["tier"].eq("in sector")
review = evaluated["tier"].eq("to review")
rows = [{"policy": "narrow_query_only", "threshold": None, **binary_metrics(y, narrow)}]
for t in THRESHOLDS:
    above = evaluated["p_cv"].ge(t)
    for policy, decision in {
        "broad_rule_a_plus_classifier": core | (review & above),
        "broad_classifier_all": (core | review) & above,
        "narrow_plus_classifier": narrow & above,
    }.items():
        rows.append({"policy": policy, "threshold": t, **binary_metrics(y, decision)})
comparison = pd.DataFrame(rows)
print(comparison.round(3).to_string(index=False))
comparison.to_csv(EXPERIMENT_DIR / "comparison_oof.csv", index=False)
evaluated["in_narrow_query"] = narrow
evaluated["positive_missed_by_narrow_query"] = y & ~narrow
evaluated.to_csv(EXPERIMENT_DIR / "labelled_decisions.csv", index=False)
print("\nPositives excluded by narrow retrieval (before classifier):", int((y & ~narrow).sum()))
print(evaluated.loc[y & ~narrow, ["docdb_family_id", "p_cv", "appln_title"]].to_string(index=False))


## 3. Coverage and full-pool volume
OOF evaluation excludes labelled families without OOF scores. Report these separately.
Production counts use the final fitted model, never for accuracy estimates.
Unscored narrow-query families remain pending; they are not silently called negatives.


In [ ]:
coverage = membership.merge(scored[["docdb_family_id", "p_sector"]], on="docdb_family_id", how="left", validate="one_to_one")
missing = coverage[coverage["p_sector"].isna()]
missing.to_csv(EXPERIMENT_DIR / "narrow_families_needing_scores.csv", index=False)
label_coverage = labels.dropna(subset=["relevance"]).copy()
label_coverage["in_narrow_query"] = label_coverage["docdb_family_id"].isin(narrow_ids)
label_coverage["has_oof_score"] = label_coverage["docdb_family_id"].isin(cv["docdb_family_id"])
label_coverage.to_csv(EXPERIMENT_DIR / "label_coverage.csv", index=False)
print("Label coverage (includes labels outside the date scope; not a recall estimate):")
print(label_coverage.groupby(["in_narrow_query", "has_oof_score", "relevance"]).size().to_string())
volumes = pd.DataFrame([{
    "threshold": t, "narrow_query_total": len(coverage),
    "narrow_with_score": int(coverage["p_sector"].notna().sum()),
    "narrow_pending_score": len(missing),
    "narrow_hybrid_accepted_scored": int(coverage["p_sector"].ge(t).sum()),
} for t in THRESHOLDS])
print("\nProduction volumes (accepted count incomplete if pending > 0):")
print(volumes.to_string(index=False))
volumes.to_csv(EXPERIMENT_DIR / "volumes.csv", index=False)
metadata = {"env": ENV, "query_commit": "c509b9b5e7de13762c832c47ea3d0ee5019b2802",
            "classifier": "existing broad-trained 07 model; OOF for evaluation",
            "evaluated_n": len(evaluated), "thresholds": THRESHOLDS,
            "pending_narrow_scores": len(missing)}
(EXPERIMENT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))
print("Saved experiment outputs to", EXPERIMENT_DIR)
